# Brain Size Data Analysis (AI)

Answers to the first SciPy statistics tutorial exercises using `brain_size.csv`.

Tutorial reference: https://scipy-lectures.org/packages/statistics/index.html

### Exercise

1. What is the mean value for VIQ of the full population?
2. How many males and females were in the study?
3. What is the average MRI count expressed in log units for males and females?

In [ ]:
# Import pandas for reading and analyzing tabular data (DataFrames).
import pandas as pd

# Import numpy for numerical operations, including the natural logarithm.
import numpy as np

# Load the brain size CSV into a DataFrame.
# sep=';' is required because columns in this file are separated by semicolons, not commas.
# na_values='.' is required because missing values in this dataset are stored as '.'.
data = pd.read_csv('brain_size.csv', sep=';', na_values='.')

# ---------------------------------------------------------------------------
# 1. Mean VIQ for the full population
# ---------------------------------------------------------------------------

# Compute the arithmetic mean of the VIQ column across all rows (all subjects).
# This answers: what is the mean VIQ of the full population?
mean_viq = data['VIQ'].mean()
print('1. Mean VIQ (full population):', mean_viq)

# ---------------------------------------------------------------------------
# 2. Number of males and females in the study
# ---------------------------------------------------------------------------

# Count how many times each Gender value appears.
# value_counts() is required to get separate counts for Male and Female.
gender_counts = data['Gender'].value_counts()
print('2. Number of males and females:')
print(gender_counts)

# ---------------------------------------------------------------------------
# 3. Average MRI count in log units for males and females
# ---------------------------------------------------------------------------

# Group rows by Gender so males and females can be summarized separately.
groupby_gender = data.groupby('Gender')

# For each gender group, take log(MRI_Count) then the mean.
# np.log(...) converts MRI counts to log units (natural log).
# .mean() is required to get the average log MRI count within each gender.
print('3. Average MRI count in log units by gender:')
for gender, value in groupby_gender['MRI_Count']:
    print((gender, np.log(value).mean()))

### Exercise: Scatter matrices by gender

Plot scatter matrices of Height vs Weight, and of PIQ vs VIQ vs FSIQ, for males and then for females.

Do the two sub-populations correspond to gender?

In [ ]:
# Import pandas plotting helpers so we can build scatter-matrix figures.
# This is required because scatter_matrix lives in pandas.plotting, not in core pandas.
from pandas import plotting

# Import matplotlib so Jupyter can display the plots created below.
# pyplot is required to label / show each figure clearly.
import matplotlib.pyplot as plt

# Reload the dataset so this cell can run on its own if needed.
# sep=';' and na_values='.' match the file format (semicolons; '.' = missing).
data = pd.read_csv('brain_size.csv', sep=';', na_values='.')

# Keep only male rows for male-specific scatter matrices.
# Boolean filtering is required to separate the male subpopulation.
male_data = data[data['Gender'] == 'Male']

# Keep only female rows for female-specific scatter matrices.
# Boolean filtering is required to separate the female subpopulation.
female_data = data[data['Gender'] == 'Female']

# ---------------------------------------------------------------------------
# Height vs Weight scatter matrices (males, then females)
# ---------------------------------------------------------------------------

# Plot Height against Weight for males as a 2x2 scatter matrix.
# A scatter matrix is required to show pairwise relationships and distributions.
plotting.scatter_matrix(male_data[['Height', 'Weight']])
plt.suptitle('Males: Height vs Weight')
plt.show()

# Plot Height against Weight for females the same way, for comparison.
plotting.scatter_matrix(female_data[['Height', 'Weight']])
plt.suptitle('Females: Height vs Weight')
plt.show()

# ---------------------------------------------------------------------------
# PIQ vs VIQ vs FSIQ scatter matrices (males, then females)
# ---------------------------------------------------------------------------

# Plot the three IQ measures against each other for males.
# Selecting PIQ, VIQ, and FSIQ is required to examine IQ relationships within males.
plotting.scatter_matrix(male_data[['PIQ', 'VIQ', 'FSIQ']])
plt.suptitle('Males: PIQ vs VIQ vs FSIQ')
plt.show()

# Plot the three IQ measures against each other for females.
# The same columns are used so male and female patterns can be compared directly.
plotting.scatter_matrix(female_data[['PIQ', 'VIQ', 'FSIQ']])
plt.suptitle('Females: PIQ vs VIQ vs FSIQ')
plt.show()

# ---------------------------------------------------------------------------
# Conclusion: do the subpopulations correspond to gender?
# ---------------------------------------------------------------------------

# Print a clear interpretation based on the IQ scatter matrices.
# In both males and females, the points still form similar high-IQ and low-IQ
# clusters. Because that two-group pattern appears inside each gender, the
# subpopulations do NOT correspond to gender.
print(
    'Conclusion: The two IQ subpopulations do NOT correspond to gender. '
    'Similar high-IQ and low-IQ clusters appear in both the male-only and '
    'female-only scatter matrices, so the grouping is within each gender '
    'rather than driven by Male vs Female.'
)

### Exercise: Hypothesis tests for Weight and VIQ

1. Test the difference between weights in males and females.
2. Use non-parametric statistics to test the difference between VIQ in males and females.

In [ ]:
# Import scipy.stats for hypothesis tests (t-tests and non-parametric tests).
# This module is required because pandas alone does not run these statistical tests.
from scipy import stats

# Reload the dataset so this cell can run on its own if needed.
# sep=';' and na_values='.' match the file format (semicolons; '.' = missing).
data = pd.read_csv('brain_size.csv', sep=';', na_values='.')

# ---------------------------------------------------------------------------
# 1. Test the difference between weights in males and females
# ---------------------------------------------------------------------------

# Extract Weight values for females only.
# Filtering by Gender is required to form the two independent samples.
female_weight = data[data['Gender'] == 'Female']['Weight']

# Extract Weight values for males only.
male_weight = data[data['Gender'] == 'Male']['Weight']

# Run an independent (two-sample) t-test comparing female vs male weights.
# ttest_ind is appropriate because weights come from different people (unpaired).
# nan_policy='omit' is required because some Weight values are missing (NaN);
# without it, the test would fail or return invalid results.
weight_ttest = stats.ttest_ind(female_weight, male_weight, nan_policy='omit')
print('1. Independent t-test: Weight (Female vs Male)')
print(weight_ttest)

# ---------------------------------------------------------------------------
# 2. Non-parametric test of VIQ difference between males and females
# ---------------------------------------------------------------------------

# Extract VIQ scores for females only.
female_viq = data[data['Gender'] == 'Female']['VIQ']

# Extract VIQ scores for males only.
male_viq = data[data['Gender'] == 'Male']['VIQ']

# Run the Mann-Whitney U test on female vs male VIQ.
# This is the non-parametric counterpart to an independent t-test:
# it does not assume the data are normally distributed (Gaussian).
# Use this instead of a t-test when you want a rank-based comparison of two groups.
viq_mannwhitney = stats.mannwhitneyu(female_viq, male_viq)
print('2. Mann-Whitney U (non-parametric): VIQ (Female vs Male)')
print(viq_mannwhitney)

# Brief interpretation helpers based on conventional alpha = 0.05.
# A small p-value means we reject the null hypothesis of no difference.
print()
print('Interpretation (alpha = 0.05):')
if weight_ttest.pvalue < 0.05:
    print('- Weight: significant difference between males and females.')
else:
    print('- Weight: no significant difference between males and females.')

if viq_mannwhitney.pvalue < 0.05:
    print('- VIQ: significant difference between males and females.')
else:
    print('- VIQ: no significant difference between males and females.')


### Exercise: Linear regression parameters

Fit a linear model of VIQ by Gender on the brain size data, retrieve the estimated parameters, and print the model summary.

In [ ]:
# Import ordinary least squares (OLS) formula API from statsmodels.
# ols is required to fit a linear regression using an R-style formula string.
from statsmodels.formula.api import ols

# Reload brain_size.csv so this cell does not depend on earlier cells.
# sep=';' and na_values='.' match the file format used throughout the tutorial.
data = pd.read_csv('brain_size.csv', sep=';', na_values='.')

# Fit a linear model: predict VIQ from Gender, including an intercept (+ 1).
# "VIQ ~ Gender + 1" means VIQ is the outcome and Gender is the predictor.
# .fit() estimates the regression coefficients from the observed data.
model = ols('VIQ ~ Gender + 1', data).fit()

# Print the estimated parameters (intercept and Gender effect).
# model.params is required to retrieve the fitted coefficient values directly.
print('Estimated parameters:')
print(model.params)
print()

# Print the full regression summary (coefficients, p-values, R-squared, etc.).
# model.summary() is required to inspect overall model fit and inference results.
print(model.summary())


### Exercise: VIQ by gender after adjusting for covariates

Test whether male and female VIQ values differ after removing the effects of brain size (`MRI_Count`), height, and weight.

In [ ]:
# Import OLS formula API (safe to re-import if this cell is run alone).
# ols is required to fit a multiple linear regression with an R-style formula.
from statsmodels.formula.api import ols

# Reload brain_size.csv so earlier cells (or other datasets) cannot overwrite this analysis.
# sep=';' and na_values='.' match the file format used in the tutorial.
data = pd.read_csv('brain_size.csv', sep=';', na_values='.')

# Fit a multiple regression of VIQ on Gender, MRI_Count, Height, and Weight.
# Including MRI_Count, Height, and Weight "removes" (adjusts for) their effects,
# so the Gender coefficient tests male vs female VIQ after those covariates.
# Rows with missing Height/Weight are dropped automatically by ols.
model = ols('VIQ ~ Gender + MRI_Count + Height + Weight', data).fit()

# Print the full model summary.
# Look at Gender[T.Male]: its coefficient and p-value answer whether VIQ still
# differs by gender after adjusting for brain size, height, and weight.
print(model.summary())
print()

# Explicitly test the null hypothesis that the Gender (Male) coefficient equals 0.
# f_test is useful as a focused test of the gender effect in this adjusted model.
print('F-test that Gender[T.Male] = 0:')
print(model.f_test('Gender[T.Male] = 0'))

# Short interpretation using alpha = 0.05.
# If the Gender p-value is large, there is no significant VIQ difference by gender
# after adjusting for MRI_Count, Height, and Weight.
gender_p = model.pvalues['Gender[T.Male]']
print()
print('Interpretation (alpha = 0.05):')
if gender_p < 0.05:
    print(
        'After adjusting for brain size, height, and weight, male and female '
        'VIQ still differ significantly.'
    )
else:
    print(
        'After adjusting for brain size, height, and weight, male and female '
        'VIQ do not differ significantly.'
    )
